# Build AlzKG

Harmonizes two edge sources into one AD-anchored knowledge graph, using the same
edge schema as PrimeKG (`relation`, `display_relation`, `x_id`, `x_type`, `x_name`,
`x_source`, `y_id`, `y_type`, `y_name`, `y_source`), so this graph can be joined
against or diffed with PrimeKG directly:

1. **Generic ontology/pathway/interaction edges** already processed by PrimeKG's
   own pipeline (MONDO, HPO, GO, Reactome, UBERON, NCBI Gene, CTD, SIDER, ...),
   read by relative path from `../../PrimeKG/datasets/data/`. Run
   `../../PrimeKG/datasets/primary_data_resources.sh` first if those files don't
   exist yet.
2. **AD-specific edges** produced by this repo's `../datasets/processing_scripts/`
   (GWAS Catalog, ClinVar, DisGeNET-AD, ClinicalTrials.gov, PharmGKB, NIAGADS,
   PubTator3 literature mining). Run `../datasets/primary_data_resources.sh` first.

The final graph is filtered to the neighborhood of the Alzheimer's disease anchor
node(s) (MONDO:0004975 and its MONDO subtypes/descendants), matching the scope
described in `../../AD_Knowledge_Graphs_Bibliography.md`.

In [ ]:
import numpy as np
import pandas as pd

primekg_data = '../../PrimeKG/datasets/data/'
ad_data = '../datasets/data/'
save_path = ad_data + 'kg/'

EDGE_COLS = ['relation', 'display_relation', 'x_id', 'x_type', 'x_name', 'x_source',
             'y_id', 'y_type', 'y_name', 'y_source']

def clean_edges(df):
    df = df.get(EDGE_COLS).dropna(subset=['x_id', 'x_type', 'y_id', 'y_type'])
    df = df.drop_duplicates()
    df = df.query('not ((x_id == y_id) and (x_type == y_type) and (x_source == y_source))')
    return df

def try_read(path, **kwargs):
    try:
        return pd.read_csv(path, low_memory=False, **kwargs)
    except FileNotFoundError:
        print(f'  [skip] not found: {path}')
        return pd.DataFrame(columns=EDGE_COLS)

## 1. Identify the Alzheimer's disease anchor node(s) in MONDO

In [ ]:
mondo_terms = try_read(primekg_data + 'mondo/mondo_terms.csv')
mondo_parents = try_read(primekg_data + 'mondo/mondo_parents.csv')

ad_mondo = mondo_terms[mondo_terms['name'].str.contains('alzheimer', case=False, na=False)] if len(mondo_terms) else pd.DataFrame()
print(f'{len(ad_mondo)} MONDO terms matching "alzheimer"')
ad_mondo

In [ ]:
# Anchor set = the matched MONDO terms plus any MONDO descendants (subtypes) of them.
AD_MONDO_ANCHOR = 'MONDO:0004975'  # Alzheimer's disease

ad_mondo_ids = set(ad_mondo['id'].astype(str)) if len(ad_mondo) else {AD_MONDO_ANCHOR.split(':')[1]}
if len(mondo_parents):
    frontier = set(ad_mondo_ids)
    while frontier:
        children = set(mondo_parents.query('parent in @frontier')['child'].astype(str))
        new = children - ad_mondo_ids
        ad_mondo_ids |= new
        frontier = new
print(f'{len(ad_mondo_ids)} MONDO ids in the AD anchor set (incl. subtypes)')

## 2. AD-specific edges (this repo's processing_scripts outputs)

In [ ]:
ad_specific_edges = []

gwas_var = try_read(ad_data + 'gwas_catalog/gwas_variant_disease.csv')
gwas_gene = try_read(ad_data + 'gwas_catalog/gwas_gene_disease.csv')
clinvar = try_read(ad_data + 'clinvar/clinvar_ad_variant_disease.csv')
disgenet_ad = try_read(ad_data + 'disgenet/disgenet_ad_gene_disease.csv')
clinicaltrials = try_read(ad_data + 'aact/clinicaltrials_drug_disease.csv')
pharmgkb = try_read(ad_data + 'pharmgkb/pharmgkb_ad_edges.csv')
niagads_var = try_read(ad_data + 'niagads/niagads_variant_disease.csv')
niagads_gene = try_read(ad_data + 'niagads/niagads_gene_disease.csv')
literature = try_read(ad_data + 'literature/literature_triples.csv')

for name, df in [('gwas_catalog_variant', gwas_var), ('gwas_catalog_gene', gwas_gene),
                  ('clinvar', clinvar), ('disgenet_ad', disgenet_ad),
                  ('clinicaltrials', clinicaltrials), ('pharmgkb', pharmgkb),
                  ('niagads_variant', niagads_var), ('niagads_gene', niagads_gene),
                  ('literature_pubtator3', literature)]:
    if len(df):
        cleaned = clean_edges(df)
        print(f'{name}: {len(cleaned)} edges')
        ad_specific_edges.append(cleaned)

ad_specific_edges = pd.concat(ad_specific_edges, ignore_index=True) if ad_specific_edges else pd.DataFrame(columns=EDGE_COLS)
print(f'\nTotal AD-specific edges: {len(ad_specific_edges)}')

## 3. Generic ontology/pathway edges from PrimeKG's pipeline, filtered to the AD neighborhood

If `../../PrimeKG/datasets/data/kg/kg.csv` already exists (i.e. you ran PrimeKG's
own `build_graph.ipynb`), pull every edge touching an AD MONDO id directly out of
it instead of re-deriving mappings by hand.

In [ ]:
primekg_kg = try_read(primekg_data + 'kg/kg.csv')

if len(primekg_kg):
    is_ad_x = primekg_kg['x_id'].astype(str).isin(ad_mondo_ids) & (primekg_kg['x_type'] == 'disease')
    is_ad_y = primekg_kg['y_id'].astype(str).isin(ad_mondo_ids) & (primekg_kg['y_type'] == 'disease')
    primekg_ad_edges = clean_edges(primekg_kg[is_ad_x | is_ad_y])
    print(f'{len(primekg_ad_edges)} PrimeKG edges touch the AD anchor set directly.')
else:
    primekg_ad_edges = pd.DataFrame(columns=EDGE_COLS)
    print('PrimeKG kg.csv not found -- run ../../PrimeKG/knowledge_graph/build_graph.ipynb first '
          'if you want the generic-ontology neighborhood included. Continuing with AD-specific '
          'edges only.')

## 4. Harmonize, assign node indices, and save `ad_kg.csv`

In [ ]:
all_edges = pd.concat([ad_specific_edges, primekg_ad_edges], ignore_index=True)
all_edges = clean_edges(all_edges)
print(f'{len(all_edges)} total harmonized edges before node indexing')

In [ ]:
# Node table: dedupe on (type, source, id) -- the same identity key PrimeKG uses
# before it merges cross-vocabulary synonyms via mondo_references/xrefs.
x_nodes = all_edges.get(['x_id', 'x_type', 'x_name', 'x_source']).rename(
    columns={'x_id': 'node_id', 'x_type': 'node_type', 'x_name': 'node_name', 'x_source': 'node_source'})
y_nodes = all_edges.get(['y_id', 'y_type', 'y_name', 'y_source']).rename(
    columns={'y_id': 'node_id', 'y_type': 'node_type', 'y_name': 'node_name', 'y_source': 'node_source'})
nodes = pd.concat([x_nodes, y_nodes], ignore_index=True).drop_duplicates(subset=['node_id', 'node_type'])
nodes = nodes.reset_index(drop=True)
nodes['node_index'] = nodes.index
print(f'{len(nodes)} unique nodes')

id_type_to_index = {(r.node_id, r.node_type): r.node_index for r in nodes.itertuples()}

ad_kg = all_edges.copy()
ad_kg['x_index'] = [id_type_to_index.get((xid, xt)) for xid, xt in zip(ad_kg['x_id'], ad_kg['x_type'])]
ad_kg['y_index'] = [id_type_to_index.get((yid, yt)) for yid, yt in zip(ad_kg['y_id'], ad_kg['y_type'])]

ad_kg = ad_kg.get(['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source',
                    'y_index', 'y_id', 'y_type', 'y_name', 'y_source'])

In [ ]:
import os
os.makedirs(save_path, exist_ok=True)
ad_kg.to_csv(save_path + 'ad_kg.csv', index=False)
nodes.to_csv(save_path + 'ad_kg_nodes.csv', index=False)
print(f'Wrote {len(ad_kg)} edges -> {save_path}ad_kg.csv')
print(f'Wrote {len(nodes)} nodes -> {save_path}ad_kg_nodes.csv')
ad_kg.head()